# **Stratified K-Fold Cross Validation**

Imbalanced datasets are common in machine learning, where one class dominates over the others. For example, in fraud detection, the number of fraudulent transactions is much smaller than legitimate transactions. Regular cross-validation techniques might not maintain the class proportions, which can lead to biased evaluation metrics.

**Stratified K-Fold Cross Validation** ensures that the class distribution in each fold is similar to the original dataset. This is crucial for imbalanced datasets to evaluate model performance fairly.

In this notebook, we will:
1. Load an imbalanced dataset.
2. Train a machine learning model.
3. Use **Stratified K-Fold Cross Validation** to evaluate the model.
4. Demonstrate how the splits preserve class proportions.


## **Step 1: Load and Explore the Dataset**

For this demonstration, we will use the **"Breast Cancer" dataset** from `sklearn.datasets`. This dataset contains features for classifying breast tumors as malignant or benign. We will artificially create an imbalance by reducing the number of samples in the minority class.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# Load the dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Convert to pandas DataFrame for visualization
df = pd.DataFrame(X, columns=data.feature_names)
df['Target'] = y

# Create an artificial imbalance by reducing the minority class
df_minority = df[df['Target'] == 1].sample(frac=0.2, random_state=42)
df_majority = df[df['Target'] == 0]
df_imbalanced = pd.concat([df_majority, df_minority])

# Update X and y
X_imbalanced = df_imbalanced.drop('Target', axis=1).values
y_imbalanced = df_imbalanced['Target'].values

# Check class distribution
print("Class Distribution in the Imbalanced Dataset:")
print(pd.Series(y_imbalanced).value_counts())


Class Distribution in the Imbalanced Dataset:
0    212
1     71
Name: count, dtype: int64


## **Step 2: Train a Random Forest Classifier**

We will use a Random Forest Classifier to demonstrate the performance of the model on the imbalanced dataset. First, we perform a simple train-test split (without cross-validation) to observe the potential bias in the results.


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imbalanced, y_imbalanced, test_size=0.3, random_state=42, stratify=y_imbalanced
)

# Train a Random Forest Classifier
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Evaluate on the test set
y_pred = clf.predict(X_test)
print("Accuracy on Test Set:", accuracy_score(y_test, y_pred))


Accuracy on Test Set: 0.9411764705882353


## **Step 3: Perform Stratified K-Fold Cross Validation**

To ensure that the class proportions are preserved in each fold, we use Stratified K-Fold Cross Validation. This will split the data into `k` folds while maintaining the distribution of the classes.


In [3]:
# Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
fold_class_distributions = []

for fold, (train_index, test_index) in enumerate(skf.split(X_imbalanced, y_imbalanced), 1):
    # Split data into training and testing sets
    X_train, X_test = X_imbalanced[train_index], X_imbalanced[test_index]
    y_train, y_test = y_imbalanced[train_index], y_imbalanced[test_index]

    # Check class distribution in the fold
    train_dist = pd.Series(y_train).value_counts(normalize=True)
    test_dist = pd.Series(y_test).value_counts(normalize=True)
    fold_class_distributions.append((train_dist, test_dist))

    # Train and evaluate the model
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy)

    print(f"Fold {fold}:")
    print("Train Class Distribution:\n", train_dist)
    print("Test Class Distribution:\n", test_dist)
    print(f"Accuracy: {accuracy}")
    print()


Fold 1:
Train Class Distribution:
 0    0.747788
1    0.252212
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.754386
1    0.245614
Name: proportion, dtype: float64
Accuracy: 0.9649122807017544

Fold 2:
Train Class Distribution:
 0    0.747788
1    0.252212
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.754386
1    0.245614
Name: proportion, dtype: float64
Accuracy: 0.8947368421052632

Fold 3:
Train Class Distribution:
 0    0.752212
1    0.247788
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.736842
1    0.263158
Name: proportion, dtype: float64
Accuracy: 0.9473684210526315

Fold 4:
Train Class Distribution:
 0    0.748899
1    0.251101
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.75
1    0.25
Name: proportion, dtype: float64
Accuracy: 0.9285714285714286

Fold 5:
Train Class Distribution:
 0    0.748899
1    0.251101
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.75
1    0.25
Name: proporti

## **Step 4: Analyze the Results**

### **1. Accuracy Across Folds**
- Stratified K-Fold ensures consistent accuracy across folds due to balanced class distributions.
- Compare the accuracy across all folds.

### **2. Class Distribution in Each Fold**
- Observe how the training and testing sets maintain the same class proportions as the original dataset.


In [4]:
# Average accuracy across folds
print("Average Accuracy Across Folds:", np.mean(fold_accuracies))

# Print class distribution for each fold
for fold, (train_dist, test_dist) in enumerate(fold_class_distributions, 1):
    print(f"Fold {fold}:")
    print("Train Class Distribution:\n", train_dist)
    print("Test Class Distribution:\n", test_dist)
    print()


Average Accuracy Across Folds: 0.9364035087719298
Fold 1:
Train Class Distribution:
 0    0.747788
1    0.252212
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.754386
1    0.245614
Name: proportion, dtype: float64

Fold 2:
Train Class Distribution:
 0    0.747788
1    0.252212
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.754386
1    0.245614
Name: proportion, dtype: float64

Fold 3:
Train Class Distribution:
 0    0.752212
1    0.247788
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.736842
1    0.263158
Name: proportion, dtype: float64

Fold 4:
Train Class Distribution:
 0    0.748899
1    0.251101
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.75
1    0.25
Name: proportion, dtype: float64

Fold 5:
Train Class Distribution:
 0    0.748899
1    0.251101
Name: proportion, dtype: float64
Test Class Distribution:
 0    0.75
1    0.25
Name: proportion, dtype: float64



# **Conclusion**

1. **Stratified K-Fold Cross Validation** ensures that class proportions are maintained across folds, making it suitable for evaluating models on imbalanced datasets.
2. In this notebook, we:
   - Created an imbalanced dataset.
   - Trained a Random Forest Classifier on the dataset.
   - Used Stratified K-Fold Cross Validation to evaluate the model.
3. **Key Observations:**
   - The class distribution in each fold is consistent with the original dataset.
   - The accuracy of the model is stable across all folds due to balanced splitting.

Stratified K-Fold is a robust technique for evaluating models on imbalanced datasets, ensuring fair evaluation and reducing bias.
